# Part 1 of 4: Collect the data (WedMeGood, Hyderabad wedding venues)

**What this notebook does:** reads the venue cards from the WedMeGood listing pages, then reads the rental cost and room count from each venue's own page. Everything is saved to `wedmegood_hyderabad_venues_raw.csv`.

**The 4 parts (run them one after the other, close each one before opening the next)**

| Part | File | Job |
|---|---|---|
| 1 | `1_collect_data.ipynb` | Collect the data (this file) |
| 2 | `2_clean_data.ipynb` | Clean the data and build the new columns |
| 3 | `3_business_analysis.ipynb` | Answer the business questions with charts |
| 4 | `4_hypothesis_tests.ipynb` | Check which differences are real, then findings and advice |

**Why this version is light on your laptop**
- It does **not** open Chrome unless the website refuses a normal download. Most of the work is plain downloads: no browser, no tabs.
- If Chrome is needed, it opens **one** window, reads one page at a time, and closes at the end of the cell.
- Work is done in **small batches** (see the settings below). Progress is saved after every page, so if anything stops you just run the cell again and it carries on.
- The slow "click +N more" step is gone. The analysis only needs the number N, and that number is already printed on the card.

**How to run:** run the cells from top to bottom. Step 3 and Step 4 can be run again and again until they say they are finished.

In [2]:
import os # Provides operating-system functions, such as working with file paths.
import re #Provides regular expressions for finding patterns in text.
import time # Provides time-related functions, such as adding delays between requests
import random # Generates random values, useful for varying delays between requests.
from datetime import date 
from urllib.parse import urljoin  # Combines a base URL with a relative URL to form a complete URL.

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://www.wedmegood.com/vendors/hyderabad/wedding-venues/"
SITE = "https://www.wedmegood.com" ## The website's root URL, used to build complete links.
TARGET_RECORDS = 500          # how many venues we want
MAX_PAGES = 40                # safety limit so the loop can never run forever
PAGES_PER_RUN = 5             # listing pages per run (about 100 venues).
DETAIL_PER_RUN = 100          # venue pages read per run in Step 4.
USE_BROWSER_IF_BLOCKED = True # Allows browser automation to be used if a normal HTTP request is blocked.
HEADLESS = True               # True = Chrome has no window (uses less memory). If Chrome keeps failing, try False
CHECKPOINT = "wedmegood_listing_checkpoint.csv"  # File used to save intermediate listing progress.
RAW_FILE = "wedmegood_hyderabad_venues_raw.csv" # File intended to store the collected raw venue data.

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
      # Indicates the kinds of responses the client says it can accept, including HTML and XML.
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-IN,en;q=0.9",
}
RENTAL_PATTERN = r"rental cost per event at .{1,150}? is\s*₹\s*([\d,]+)"
ROOMS_PATTERN = r"offers\s+(\d+)\s+well-appointed rooms"

session = requests.Session()  # Creates a reusable HTTP session for making multiple requests.
session.headers.update(HEADERS)  # Applies the specified headers to requests made through this session.
print("Setup done.") # Displays a message confirming that this setup section has executed.

Setup done.


## Step 1: Helper functions
These are only definitions. Nothing is downloaded in this cell.

In [3]:
# Prepare functions to fetch pages, parse cards, save progress and manage Chrome.
browser = {"driver": None}  # Stores the Selenium Chrome driver; None means Chrome has not been opened.


def load_records():
    """Read the saved progress file (empty list if nothing is saved yet). Every step starts from this file,
    so the notebook still works after a restart."""
    if not os.path.exists(CHECKPOINT):  # Check whether the checkpoint CSV file exists.
        return []  # If it doesn't exist, return an empty list because no saved records are available.
    saved = pd.read_csv(CHECKPOINT)  # Read the saved CSV file into a pandas DataFrame.
    # Convert each row into a dictionary and replace missing pandas values (NaN) with None.
    return [{k: (None if pd.isna(v) else v) for k, v in row.items()} for row in saved.to_dict("records")] 
    # This converts every row of the CSV into a dictionary and replaces missing values (NaN) with Python's None.


def open_browser():
    """Start ONE light Chrome. Selenium is imported here so the notebook works even if Selenium is not installed."""
    from selenium import webdriver  # Import Selenium's browser-control tools only when this function is called.
    options = webdriver.ChromeOptions()  # Create a Chrome configuration object.
    options.page_load_strategy = "eager"  # Continue when the main page content is loaded; don't wait for every resource.
    if HEADLESS:  # Check whether Chrome should run without displaying its window.
        options.add_argument("--headless=new")  # Run Chrome in headless mode, without a visible window.
    options.add_argument("--window-size=1200,900")  # Set the browser window dimensions.
    options.add_argument("--disable-gpu")  # Disable GPU acceleration; can help in some environments.
    options.add_argument("--disable-extensions")  # Prevent Chrome extensions from loading.
    options.add_argument("--disable-dev-shm-usage")  # Avoid using the limited shared-memory area in some environments.
    options.add_argument("--blink-settings=imagesEnabled=false")  # Ask Chrome not to load images.
    options.add_argument("--disable-blink-features=AutomationControlled")  # Modify an automation-related browser feature; this does not guarantee that automation will go undetected.
    # Set Chrome's content preferences so images are blocked.
    options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    browser["driver"] = webdriver.Chrome(options=options)  # Launch Chrome and store its driver in the shared dictionary.
    return browser["driver"]  # Return the driver so other functions can control the browser.


def close_browser():
    """Close Chrome if it is open (safe to call any time)."""
    if browser["driver"] is not None:  # Check whether a Chrome driver has been stored.
        try:
            browser["driver"].quit()  # Close Chrome and end the WebDriver session.
        except Exception:
            pass  # Ignore errors that occur while closing Chrome.
        browser["driver"] = None  # Reset the stored driver to indicate that Chrome is closed.


def save_debug(html, note=""):
    """Save what the website really sent, so you can see why no venue cards were found."""
    with open("debug_page.html", "w", encoding="utf-8") as f:  # Open/create an HTML file for debugging, using UTF-8 encoding.
        f.write(html or "")  # Write the HTML into the file; use an empty string if html is None or empty.
    soup = BeautifulSoup(html or "", "html.parser")  # Parse the HTML so its title and text can be inspected.
    title = soup.title.get_text(strip=True) if soup.title else "(no title)"  # Extract the page title, or use a fallback if absent.
    snippet = soup.get_text(" ", strip=True)[:200]  # Extract visible text, normalize spaces, and keep only the first 200 characters.
    print(f"  {note}Page title: {title}")  # Print the supplied note and the page title.
    print(f"  First words on the page: {snippet}")  # Print a short text preview to help identify the page response.
    print("  The full page was saved as debug_page.html")  # Tell you where the complete HTML was saved.


def fetch_cards(page):
    """Return (list of venue cards, how they were fetched) for one listing page."""
    url = f"{BASE_URL}?page={page}"  # Build the listing-page URL for the requested page number.
    status = None  # Initialize the HTTP status or error information.
    try:
        resp = session.get(url, timeout=25)  # Request the page using the existing session; stop waiting after 25 seconds.
        status = resp.status_code  # Store the HTTP response status code, such as 200 or 403.
        if status == 200:  # Check whether the server returned a successful HTTP response.
            # Parse the downloaded HTML and select elements matching the CSS class "vendor-card".
            cards = BeautifulSoup(resp.text, "html.parser").select(".vendor-card")
            if cards:  # Check whether any venue cards were found.
                return cards, "plain download"  # Return the cards and identify the method used to fetch them.
            # Save the response for inspection if the request succeeded but no cards were found.
            save_debug(resp.text, "Plain download worked but had no venue cards. ")
    except requests.RequestException as err:  # Catch request-related errors, such as timeouts or connection failures.
        status = type(err).__name__  # Store the error's class name instead of an HTTP status code.

    if not USE_BROWSER_IF_BLOCKED:  # Check whether browser fallback has been disabled.
        return [], f"no cards (status {status})"  # Return no cards and explain the recorded status/error.

    # Inform you that the normal request found no cards and the code will try Selenium.
    print(f"  Plain download gave no cards (status {status}). Trying one Chrome window...")
    try:
        from selenium.webdriver.common.by import By  # Provides locator types, such as CSS selectors.
        from selenium.webdriver.support.ui import WebDriverWait  # Allows the code to wait for a browser condition.
        from selenium.webdriver.support import expected_conditions as EC  # Provides predefined conditions for explicit waits.
        driver = browser["driver"] or open_browser()  # Reuse the existing driver, or open Chrome if none exists.
        driver.get(url)  # Navigate Chrome to the listing page.
        # Wait up to 30 seconds until at least one element matching ".vendor-card" appears.
        WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vendor-card")))
        time.sleep(1)  # Wait one additional second for the page to settle.
        # Parse the HTML currently loaded in Chrome and extract the venue-card elements.
        cards = BeautifulSoup(driver.page_source, "html.parser").select(".vendor-card")
        return cards, "Chrome"  # Return the cards and indicate that Chrome fetched the page.
    except Exception as err:  # Catch errors during browser setup, page loading, waiting, or parsing.
        print(f"  Chrome could not read the page: {type(err).__name__}")  # Display the error's class name.
        if isinstance(err, ModuleNotFoundError):  # Check whether a required Python module could not be imported.
            # Explain how to install Selenium if it is missing.
            print("  Selenium is not installed. Run this once in a terminal, then try again:  pip install selenium")
        if browser["driver"] is not None:  # Check whether a Chrome driver exists for debugging.
            try:
                save_debug(browser["driver"].page_source, "")  # Save Chrome's current page HTML to inspect the failure.
            except Exception:
                pass  # Ignore errors if the browser's HTML cannot be retrieved.
        return [], f"failed ({type(err).__name__})"  # Return an empty card list and the failure type.


def parse_card(card, page):
    """Turn one venue card into one row (a dictionary)."""
    text = card.get_text(" ", strip=True)  # Extract all text from the card and normalize whitespace.

    name_tag = card.select_one("a.vendor-detail")  # Find the first link matching the CSS selector "a.vendor-detail".
    # Extract the venue name if the link exists; otherwise, store None.
    name = name_tag.get_text(" ", strip=True) if name_tag is not None else None
    if name_tag is not None and name_tag.has_attr("href"):  # Check that the name link exists and has an href attribute.
        link_tag = name_tag  # Use the venue-name link to obtain the detail-page URL.
    else:
        # If the preferred link is unavailable, look for a link containing "/wedding-venues/" in its href.
        link_tag = card.select_one("a[href*='/wedding-venues/']")
    # Convert the relative link into a complete URL using SITE; return None if no suitable link exists.
    link = urljoin(SITE, link_tag["href"]) if link_tag is not None else None

    rating_tag = card.select_one("span.StarRatingNew")  # Find the element that contains the venue rating.
    review_tag = card.select_one("span.review-cnt")  # Find the element that contains the review count.
    # Extract text from each div.line element; the resulting list may contain location and other card details.
    lines = [div.get_text(" ", strip=True) for div in card.select("div.line")]

    # Initialize all three price variables as None because the card may not show every price type.
    veg_price, nonveg_price, function_price = None, None, None
    # Find price text matching an optional label, rupee amount, and unit (plate or function).
    for label, amount, unit in re.findall(r"(Non veg|Veg)?\s*₹\s*([\d,]+)\s*per\s+(plate|function)", text):
        if unit == "plate" and label == "Veg":  # Check whether the price is for a vegetarian plate.
            veg_price = amount  # Save the vegetarian per-plate price as text.
        elif unit == "plate" and label == "Non veg":  # Check whether the price is for a non-vegetarian plate.
            nonveg_price = amount  # Save the non-vegetarian per-plate price as text.
        elif unit == "function" and function_price is None:  # Check for a function price, keeping only the first match.
            function_price = amount  # Save the venue's per-function price as text.

    capacity = re.search(r"(\d[\d,]*)-(\d[\d,]*)\s*pax", text)  # Find a capacity range such as "300-1000 pax".
    rooms = re.search(r"(\d+)\s*Rooms", text)  # Find a room count such as "30 Rooms".
    more = re.search(r"\+\s*(\d+)\s*more", text)  # Find text such as "+3 more" and capture the number.
    # Find Indoor/Outdoor labels, remove duplicates with set(), then sort the unique values.
    spaces = sorted(set(re.findall(r"Indoor|Outdoor", text)))

    return {
        "venue_name": name,  # Store the extracted venue name.
        "detail_url": link,  # Store the venue's detail-page URL.
        "page": page,  # Store the listing-page number where the venue was found.
        "is_paid": "Paid Vendor" in text,  # Store True if that exact phrase appears in the card text; otherwise False.
        "rating": rating_tag.get_text(" ", strip=True) if rating_tag is not None else None,  # Store the rating or None.
        "reviews": review_tag.get_text(" ", strip=True) if review_tag is not None else None,  # Store the review count or None.
        "location_text": lines[1] if len(lines) > 1 else None,  # Store the second extracted line if available; otherwise None.
        "space_type": " & ".join(spaces) if spaces else None,  # Join unique space labels, e.g. "Indoor & Outdoor", or store None.
        "veg_price": veg_price,  # Store the vegetarian per-plate price.
        "nonveg_price": nonveg_price,  # Store the non-vegetarian per-plate price.
        "venue_price_card": function_price,  # Store the per-function price found on the listing card.
        "capacity_min": capacity.group(1) if capacity else None,  # Store the lower capacity bound if a range was found.
        "capacity_max": capacity.group(2) if capacity else None,  # Store the upper capacity bound if a range was found.
        "rooms_card": rooms.group(1) if rooms else None,  # Store the room count found on the card.
        "more_count": more.group(1) if more else None,  # Store the number shown in the "+N more" text.
        "rental_cost_detail": None,   # Placeholder; the venue detail page will fill this in Step 4.
        "rooms_detail": None,         # Placeholder; the detail-page room count will be filled in Step 4.
        "detail_status": None,        # Placeholder for the detail-page scraping status, filled in Step 4.
    }


print("Helper functions are ready.")  # Confirm that the function definitions have been executed.

Helper functions are ready.


## Step 2: Quick test (one page only)
Run this once before the big loop. It reads **page 1**, tells you how many venue cards it found, and shows one parsed card.

- If you see about 20 cards and a sensible sample row, go on to Step 3.
- If you see **0 cards**, read the lines printed above the result. They show the page title and first words the website sent (for example "Access Denied" or a captcha). Then try: set `HEADLESS = False` in the settings cell and run again, wait a few minutes, or try another network.

In [5]:
cards, how = fetch_cards(1)  # Fetch page 1 and store the venue cards in cards and the fetching method in how.

close_browser()  # Always close Chrome after the test to release browser resources.

print("Method used:", how)  # Display how the page was fetched, such as "plain download" or "Chrome".

print("Venue cards found on page 1:", len(cards))  # Count and display the number of venue cards found.

if cards:  # Check whether the cards list contains at least one venue card.
    sample = parse_card(cards[0], 1)  # Extract details from the first card and store them in a dictionary.
    for key, value in sample.items():  # Loop through each field name and its extracted value.
        print(f"  {key:20} {value}")  # Print the field name in a 20-character-wide space, followed by its value.
else:  # Execute this block if no venue cards were found.
    print("No cards found. See the notes above this result.")  # Display a message indicating that extraction found no cards.

Method used: plain download
Venue cards found on page 1: 20
  venue_name           Advaya Convention
  detail_url           https://www.wedmegood.com/wedding-venues/Advaya-Convention-26351086
  page                 1
  is_paid              False
  rating               5.0
  reviews              ( 7 reviews )
  location_text        Medchal , Hyderabad Large Lawn with Banquet
  space_type           None
  veg_price            None
  nonveg_price         None
  venue_price_card     None
  capacity_min         500
  capacity_max         10000
  rooms_card           8
  more_count           3
  rental_cost_detail   None
  rooms_detail         None
  detail_status        None


## Step 3: Collect the listing pages (small batches)
**Run this cell, wait for it to finish, then run it again until it says `500 venues collected`.**
Each run reads up to `PAGES_PER_RUN` pages, saves after every page, and closes Chrome (if it was opened) at the end.
If the laptop hangs: close Chrome, restart the kernel, run the setup and helper cells again, and run this cell. It continues from the saved file.

In [13]:
records = load_records()  # Load previously saved venue records from the checkpoint CSV.

seen_urls = {r["detail_url"] for r in records if r.get("detail_url")}  # Create a set of existing venue URLs to avoid duplicates.

# Find the highest page number already saved, then start from the next page.
page = int(max((int(r.get("page") or 0) for r in records), default=0)) + 1

pages_done = 0  # Track how many listing pages have been processed during this run.

# Display the starting page and the number of records already saved.
print(f"Starting at page {page} with {len(records)} venues already saved")

try:
    # Continue while all three conditions are satisfied:
    # target not reached, page limit not exceeded, and per-run page limit not reached.
    while len(records) < TARGET_RECORDS and page <= MAX_PAGES and pages_done < PAGES_PER_RUN:

        cards, how = fetch_cards(page)  # Fetch venue cards from the current listing page and record the method used.

        if not cards:  # Check whether the page returned any venue cards.
            print(f"Page {page}: no cards found ({how}). Stopping this run.")  # Explain why processing is stopping.
            break  # Exit the while loop because no cards were found.

        added = 0  # Count the new venue records added from the current page.

        for card in cards:  # Process each venue card found on this page.

            if len(records) >= TARGET_RECORDS:  # Check whether the target number of venues has been reached.
                break  # Stop processing more cards when the target is reached.

            try:
                row = parse_card(card, page)  # Extract the current card's details into a dictionary.
            except Exception as err:  # Handle an error that occurs while parsing an individual card.
                print(f"  One card on page {page} was skipped ({type(err).__name__})")  # Display the error type.
                continue  # Skip this card and move to the next one.

            # If the venue has a URL and that URL was already collected, skip the duplicate.
            if row["detail_url"] and row["detail_url"] in seen_urls:
                continue  # Promoted venues can appear more than once, so skip repeated URLs.

            if row["detail_url"]:  # Check whether this venue has a detail-page URL.
                seen_urls.add(row["detail_url"])  # Add the URL to the set so future duplicates can be detected.

            records.append(row)  # Add the venue's extracted details to the main records list.
            added += 1  # Increase the count of new venues collected from this page.

        # Display the page number, cards found, new venues added, total collected, and fetching method.
        print(f"Page {page}: {len(cards)} cards, {added} new venues, {len(records)} collected so far ({how})")

        pd.DataFrame(records).to_csv(CHECKPOINT, index=False)  # Save all collected records after every page; omit the CSV index.

        page += 1  # Move to the next listing page.
        pages_done += 1  # Record that one more page has been processed.

        time.sleep(random.uniform(1, 2))  # Pause for a random 1–2 seconds before requesting another page.

finally:
    close_browser()  # Always attempt to close Chrome, even if an error occurs or the loop stops.

# Check whether the desired number of venue records has been collected.
if len(records) >= TARGET_RECORDS:
    print(f"{len(records)} venues collected. Go to Step 4.")  # Tell you the target is reached and the next step can begin.
else:
    print(f"{len(records)} venues collected. Run this cell again to continue.")  # Tell you to rerun the cell if more records are needed.

Starting at page 26 with 483 venues already saved
Page 26: 20 cards, 17 new venues, 500 collected so far (plain download)
500 venues collected. Go to Step 4.


## Step 4: Read the rental cost and rooms from each venue's own page
Each venue page has a short question-and-answer part, for example "The rental cost per event at *venue* is ₹45,000." This fills the gaps for venues whose card shows no flat price.

This step uses plain downloads only (no browser). It reads up to `DETAIL_PER_RUN` pages per run. **Run it again until it says there is nothing left.**
Venues marked `http_error` are tried again on the next run. Venues marked `no_sentence` simply do not have that sentence, which is normal.

In [29]:
records = load_records()  # Load all venue records saved in the checkpoint CSV.

if not records:  # Check whether the records list is empty.
    raise SystemExit("No saved venues yet. Run Step 3 first.")  # Stop execution if Step 3 has not saved any venues.

# Select venues that have a detail URL and whose detail pages still need processing.
# Records marked "ok" or "no_sentence" are excluded because they have already been processed.
todo = [r for r in records if r.get("detail_url") and r.get("detail_status") not in ("ok", "no_sentence")]

batch = todo[:DETAIL_PER_RUN]  # Select only the first DETAIL_PER_RUN venues for this run.

# Display the total number of venues needing processing and how many will be processed now.
print(f"Venue pages still to read: {len(todo)}. Reading {len(batch)} now.")

# Process each venue record in the selected batch, numbering the records from 1.
for k, record in enumerate(batch, start=1):
    try:
        # Send an HTTP request to the venue's detail page, with a 25-second timeout.
        resp = session.get(record["detail_url"], timeout=25)

        if resp.status_code != 200:  # Check whether the server returned a successful HTTP response.
            record["detail_status"] = "http_error"  # Mark the detail page as having an HTTP error.
        else:
            # Parse the downloaded HTML and extract its text, joining text elements with spaces.
            page_text = BeautifulSoup(resp.text, "html.parser").get_text(" ", strip=True)

            # Search the page text for the rental-cost sentence using the predefined regex pattern.
            rental = re.search(RENTAL_PATTERN, page_text, re.IGNORECASE)

            # Search the page text for the room-count sentence using the predefined regex pattern.
            rooms = re.search(ROOMS_PATTERN, page_text, re.IGNORECASE)

            # Store the captured rental amount if a matching sentence was found; otherwise store None.
            record["rental_cost_detail"] = rental.group(1) if rental else None

            # Store the captured room count if a matching sentence was found; otherwise store None.
            record["rooms_detail"] = rooms.group(1) if rooms else None

            # Mark as "ok" if the rental sentence was found; otherwise mark "no_sentence".
            record["detail_status"] = "ok" if rental else "no_sentence"

    except Exception:  # Catch unexpected errors while requesting or processing this venue page.
        record["detail_status"] = "http_error"  # Mark the record as having an error.

    if k % 25 == 0:  # Check whether 25 venue pages have been processed in this batch.
        print(f"  {k} venue pages read")  # Display progress after every 25 records.
        pd.DataFrame(records).to_csv(CHECKPOINT, index=False)  # Save all records to the checkpoint CSV.

    time.sleep(random.uniform(0.5, 1.0))  # Pause for a random 0.5–1 second before processing the next venue.

# Save all records after the batch finishes, including updates to the final records in the batch.
pd.DataFrame(records).to_csv(CHECKPOINT, index=False)

# Count the different detail-processing statuses, including missing statuses represented by NaN.
print(pd.Series([r.get("detail_status") for r in records]).value_counts(dropna=False))

# Count venues that have a detail URL but are not yet marked "ok" or "no_sentence".
left = len([r for r in records if r.get("detail_url") and r.get("detail_status") not in ("ok", "no_sentence")])

# If no eligible venue pages remain, announce the next step; otherwise, report how many remain.
print("Nothing left to read. Go to Step 5." if left == 0 else f"{left} venue pages left. Run this cell again.")

Venue pages still to read: 1. Reading 1 now.
no_sentence    362
ok             138
Name: count, dtype: int64
Nothing left to read. Go to Step 5.


## Step 5: Save the raw file
This creates `wedmegood_hyderabad_venues_raw.csv`, which Part 2 reads. You do not need to scrape again after this.

In [4]:
records = load_records()  # Load the saved venue records from the checkpoint CSV.

if not records:  # Check whether the records list is empty.
    raise SystemExit("No saved venues yet. Run Step 3 first.")  # Stop execution if no venue records have been saved.

df_raw = pd.DataFrame(records)  # Convert the list of venue dictionaries into a pandas DataFrame.
df_raw.to_csv(RAW_FILE, index=False)  # Save the DataFrame to the raw-data CSV file without adding a row-index column.

print("Scraped on:", date.today())  # Display today's date as the scraping/export date.

print("Saved:", RAW_FILE, "| rows and columns:", df_raw.shape)  # Display the output filename and DataFrame dimensions (rows, columns).

# Count how many venues have a non-missing flat/function price on the listing card.
print("Venues with a flat price on the card:", df_raw["venue_price_card"].notna().sum())

# Count how many venues have a non-missing rental cost extracted from their detail pages.
print("Venues with a rental cost from the venue page:", df_raw["rental_cost_detail"].notna().sum())

if len(df_raw) < TARGET_RECORDS:  # Check whether the number of collected venues is below the target.
    # Inform you that you can proceed or return to Step 3 to collect more venues.
    print(f"Only {len(df_raw)} venues so far. You can still continue to Part 2, or go back to Step 3 for more.")
else:
    print("Done. Close this notebook and open Part 2.")  # Confirm that the target has been reached and indicate the next step.

Scraped on: 2026-10-10
Saved: wedmegood_hyderabad_venues_raw.csv | rows and columns: (500, 18)
Venues with a flat price on the card: 89
Venues with a rental cost from the venue page: 138
Done. Close this notebook and open Part 2.
